In [1]:
#pip install imageio

In [ ]:
# 导入必要的库
import os
import time
import random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import matplotlib.pyplot as plt
import matplotlib.patches as mpts
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, cohen_kappa_score, accuracy_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import minmax_scale
import pandas as pd
from scipy.io import loadmat
from tqdm.notebook import tqdm
from IPython import display
import h5py
import copy
import sys
import glob
import seaborn as sns
from datetime import datetime
%matplotlib inline

In [ ]:
## 超参数和实验设置配置单元格

# 设置随机种子，确保实验可重复性
RANDOM_SEED = 666

# 定义模型名称，用于结果保存和模型标识
MODEL_NAME = 'BrainVoxel_1DKAN'

# 指定数据集名称
DATASET = 'BrainVoxel'

# 数据集划分比例
TRAIN_RATE = 0.7  # 将原训练集按7:3分割，70%用于训练
TEST_RATE = 0.3   # 30%用于测试
# 原验证集保持不变，用作最终验证

# 训练参数
EPOCH = 100        # 总训练轮数
VAL_EPOCH = 1      # 每隔多少轮进行一次验证
LR = 0.001         # 学习率
WEIGHT_DECAY = 1e-6  # 权重衰减系数，用于L2正则化
BATCH_SIZE = 64    # 批处理大小，固定不变

# 计算设备选择
DEVICE = 0         # -1表示使用CPU，0表示使用第一块GPU(cuda:0)

# 数据参数
FEATURE_DIM = 341  # 输入特征维度
NUM_CLASS = 2      # 二分类问题：正类和负类
FIXED_GRID = 3     # 固定网格大小，不进行网格扩展

# 数据预处理参数
N_PCA = 15         # PCA降维后保留的主成分数量，0表示使用原始数据不进行降维
NORM = True        # 是否对数据进行标准化/归一化处理

# 模型检查点路径
CHECK_POINT = None  # 加载预训练模型的路径，None表示从头开始训练

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
# 如果保存目录不存在，则创建该目录
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [ ]:
## 设置随机数种子，确保实验结果可复现

# 为Python的random模块设置随机种子
random.seed(RANDOM_SEED)

# 为PyTorch的CPU操作设置随机种子
torch.manual_seed(RANDOM_SEED)

# 为当前GPU设置随机种子
torch.cuda.manual_seed(RANDOM_SEED)

# 为所有可用GPU设置相同的随机种子
torch.cuda.manual_seed_all(RANDOM_SEED)

# 为NumPy库设置随机种子
np.random.seed(RANDOM_SEED)

# 禁用CuDNN的非确定性算法
torch.backends.cudnn.deterministic = True

# 禁用CuDNN的自动优化选择
torch.backends.cudnn.benchmark = False

In [ ]:
# 脑体素数据载入工具函数
def load_brain_voxel_data():
    """
    载入脑体素数据、标签和训练/验证/测试集
    
    返回:
        data: 高维特征数据
        train_gt: 训练集标签
        val_gt: 验证集标签
        all_data_dict: 包含所有数据集信息的字典
    """
    # 路径配置 - 根据您的实际路径进行调整
    output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output'
    train_label_dir = os.path.join(output_path, 'train_set_by_label')
    val_label_dir = os.path.join(output_path, 'val_set_by_label')
    
    # 读取标签索引文件
    def load_label_index(index_file):
        label_info = {}
        with open(index_file, 'r') as f:
            # 跳过表头
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    label_id = int(parts[0])
                    voxel_count = int(parts[1])
                    filename = parts[2] if parts[2] else None
                    label_info[label_id] = {'count': voxel_count, 'filename': filename}
        return label_info
    
    train_index_file = os.path.join(train_label_dir, "label_index.txt")
    val_index_file = os.path.join(val_label_dir, "val_label_index.txt")
    
    if not os.path.exists(train_index_file):
        raise FileNotFoundError(f"训练标签索引文件不存在: {train_index_file}")
    if not os.path.exists(val_index_file):
        raise FileNotFoundError(f"验证标签索引文件不存在: {val_index_file}")
    
    train_label_info = load_label_index(train_index_file)
    val_label_info = load_label_index(val_index_file)
    
    # 获取有效标签（有体素数据的标签）
    valid_labels = [label_id for label_id, info in train_label_info.items() 
                   if info['count'] > 0]
    
    print(f"数据加载完成: 找到 {len(valid_labels)} 个有效标签")
    
    return {
        'train_label_info': train_label_info,
        'val_label_info': val_label_info,
        'valid_labels': valid_labels,
        'train_label_dir': train_label_dir,
        'val_label_dir': val_label_dir
    }

# 加载数据集信息
all_data_dict = load_brain_voxel_data()

In [ ]:
def apply_pca(X, num_components=15, norm=True):
    """
    对数据进行PCA降维和标准化处理
    
    参数:
        X (ndarray): 需要降维的数据
        num_components (int): 保留的主成分数量，0表示不进行PCA
        norm (bool): 是否进行标准化处理
    
    返回:
        new_X: 处理后的数据
        num_components: 最终的特征维度
    """
    if num_components == 0:
        new_X = np.reshape(X, (-1, X.shape[2]))
    else:
        new_X = np.reshape(X, (-1, X.shape[2]))
        pca = PCA(n_components=num_components)
        new_X = pca.fit_transform(new_X)
    
    if norm:
        new_X = minmax_scale(new_X, axis=1)
    
    new_X = np.reshape(new_X, (X.shape[0], X.shape[1], -1))
    return new_X, new_X.shape[2]

In [ ]:
class BrainVoxelDataset(Dataset):
    """
    脑体素数据集类，简化版
    """
    def __init__(self, data, labels, is_inference=False):
        """
        初始化数据集
        
        参数:
            data: 特征数据，形状为(n_samples, feature_dim)
            labels: 标签数据，形状为(n_samples,)
            is_inference: 是否为推理模式（不返回标签）
        """
        super(BrainVoxelDataset, self).__init__()
        self.data = data
        self.labels = labels
        self.is_inference = is_inference
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx]
        x = torch.FloatTensor(x)
        
        if self.is_inference:
            return x
        else:
            y = self.labels[idx]
            y = torch.LongTensor([y])[0]  # 转为长整型标量
            return x, y

In [ ]:
def get_dataset_for_label(label_id, all_data_dict, train_test_split_ratio=0.7):
    """
    获取指定标签的数据集，并按比例分割为训练集和测试集
    
    参数:
        label_id: 目标标签ID
        all_data_dict: 包含数据路径和信息的字典
        train_test_split_ratio: 训练集占原训练数据的比例
    
    返回:
        dataset_dict: 包含训练集、测试集和验证集的字典
    """
    train_label_dir = all_data_dict['train_label_dir']
    val_label_dir = all_data_dict['val_label_dir']
    valid_labels = all_data_dict['valid_labels']
    
    if label_id not in valid_labels:
        raise ValueError(f"标签 {label_id} 不在有效标签列表中")
    
    # 获取目标标签的文件路径
    def get_label_file_path(label_id, is_validation=False):
        if is_validation:
            pattern = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
        else:
            pattern = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
        
        matches = glob.glob(pattern)
        return matches[0] if matches else None
    
    # 加载正样本（目标标签的体素）
    train_file = get_label_file_path(label_id, False)
    val_file = get_label_file_path(label_id, True)
    
    if not train_file:
        raise ValueError(f"标签 {label_id} 没有训练数据文件")
    
    # 加载训练数据（正样本）
    positive_samples = np.load(train_file)
    positive_labels = np.ones(len(positive_samples))
    
    # 获取负样本（来自其他标签）
    negative_samples = []
    other_labels = [l for l in valid_labels if l != label_id]
    random.shuffle(other_labels)
    
    # 保持与正样本相同数量的负样本
    target_negative_count = len(positive_samples)
    current_negative_count = 0
    
    for other_label in other_labels:
        if current_negative_count >= target_negative_count:
            break
        
        other_file = get_label_file_path(other_label, False)
        if not other_file:
            continue
        
        other_samples = np.load(other_file)
        
        # 限制从每个标签获取的负样本数量
        samples_to_take = min(len(other_samples), 
                             target_negative_count - current_negative_count)
        
        if samples_to_take < len(other_samples):
            # 随机抽取部分样本
            indices = np.random.choice(len(other_samples), samples_to_take, replace=False)
            selected_samples = other_samples[indices]
        else:
            selected_samples = other_samples
        
        negative_samples.append(selected_samples)
        current_negative_count += len(selected_samples)
    
    if negative_samples:
        negative_samples = np.vstack(negative_samples)
        # 如果收集到的负样本超过需求，再次随机抽取
        if len(negative_samples) > target_negative_count:
            indices = np.random.choice(len(negative_samples), target_negative_count, replace=False)
            negative_samples = negative_samples[indices]
    else:
        negative_samples = np.array([]).reshape(0, positive_samples.shape[1])
    
    negative_labels = np.zeros(len(negative_samples))
    
    # 合并正负样本
    all_train_samples = np.vstack([positive_samples, negative_samples])
    all_train_labels = np.concatenate([positive_labels, negative_labels])
    
    # 随机打乱并分割训练/测试集
    indices = np.arange(len(all_train_samples))
    np.random.shuffle(indices)
    all_train_samples = all_train_samples[indices]
    all_train_labels = all_train_labels[indices]
    
    # 按比例分割为训练集和测试集
    split_idx = int(len(all_train_samples) * train_test_split_ratio)
    train_samples = all_train_samples[:split_idx]
    train_labels = all_train_labels[:split_idx]
    test_samples = all_train_samples[split_idx:]
    test_labels = all_train_labels[split_idx:]
    
    # 加载验证集（如果有）
    val_samples = []
    val_labels = []
    
    if val_file:
        pos_val_samples = np.load(val_file)
        pos_val_labels = np.ones(len(pos_val_samples))
        
        # 获取验证集负样本
        val_negative_samples = []
        val_target_count = len(pos_val_samples)
        val_current_count = 0
        
        for other_label in other_labels:
            if val_current_count >= val_target_count:
                break
                
            other_val_file = get_label_file_path(other_label, True)
            if not other_val_file:
                continue
                
            other_val_samples = np.load(other_val_file)
            samples_to_take = min(len(other_val_samples), 
                                 val_target_count - val_current_count)
            
            if samples_to_take < len(other_val_samples):
                indices = np.random.choice(len(other_val_samples), samples_to_take, replace=False)
                selected_val_samples = other_val_samples[indices]
            else:
                selected_val_samples = other_val_samples
                
            val_negative_samples.append(selected_val_samples)
            val_current_count += len(selected_val_samples)
        
        if val_negative_samples:
            val_negative_samples = np.vstack(val_negative_samples)
            if len(val_negative_samples) > val_target_count:
                indices = np.random.choice(len(val_negative_samples), val_target_count, replace=False)
                val_negative_samples = val_negative_samples[indices]
                
            neg_val_labels = np.zeros(len(val_negative_samples))
            
            # 合并验证集的正负样本
            val_samples = np.vstack([pos_val_samples, val_negative_samples])
            val_labels = np.concatenate([pos_val_labels, neg_val_labels])
            
            # 随机打乱验证集
            val_indices = np.arange(len(val_samples))
            np.random.shuffle(val_indices)
            val_samples = val_samples[val_indices]
            val_labels = val_labels[val_indices]
    
    print(f"数据集创建完成:")
    print(f"  训练集: {len(train_samples)} 样本")
    print(f"  测试集: {len(test_samples)} 样本")
    print(f"  验证集: {len(val_samples)} 样本")
    
    return {
        'train_samples': train_samples,
        'train_labels': train_labels,
        'test_samples': test_samples, 
        'test_labels': test_labels,
        'val_samples': val_samples,
        'val_labels': val_labels
    }

In [ ]:
from fast_kan import FastKAN

class BrainVoxelKAN(nn.Module):
    """
    用于脑体素分类的KAN模型，简化版
    """
    def __init__(self, input_dim, hidden_dim, num_classes, grid_size=3):
        """
        初始化模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏层维度
            num_classes: 类别数量
            grid_size: 网格大小，保持固定
        """
        super(BrainVoxelKAN, self).__init__()
        
        self.kan = FastKAN(
            layers_hidden=[input_dim, hidden_dim, num_classes],
            num_grids=grid_size
        )
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征，形状为(batch_size, input_dim)
            
        返回:
            output: 模型输出，形状为(batch_size, num_classes)
        """
        return self.kan(x)

In [ ]:
def train_brain_voxel_kan(model, train_loader, test_loader, criterion, optimizer, device, 
                          num_epochs=100, val_epoch=1, save_path="./Results"):
    """
    训练脑体素KAN模型，与1D KAN训练流程保持一致
    
    参数:
        model: KAN模型
        train_loader: 训练数据加载器
        test_loader: 测试数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 模型保存路径
    
    返回:
        训练结果统计信息
    """
    # 初始化统计变量
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    test_num = len(test_loader.dataset)
    
    try:
        # 训练循环
        for e in tqdm(range(num_epochs), desc="Training:"):
            # 设置模型为训练模式
            model.train()
            avg_loss = 0.0
            train_acc = 0
            
            # 批次循环
            for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=batch_num):
                # 将数据移动到指定设备
                data, target = data.to(device), target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out = model(data)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失和准确率
                avg_loss += loss.item()
                _, pred = torch.max(out, dim=1)
                train_acc += (pred == target).sum().item()
            
            # 计算本轮平均损失和准确率
            loss_list.append(avg_loss / train_num)
            acc_list.append(train_acc / train_num)
            print(f"epoch {e}/{num_epochs} loss:{loss_list[-1]}  acc:{acc_list[-1]}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                val_acc = 0
                model.eval()
                
                with torch.no_grad():
                    for batch_idx, (data, target) in tqdm(enumerate(test_loader), total=len(test_loader)):
                        data, target = data.to(device), target.to(device)
                        out = model(data)
                        _, pred = torch.max(out, dim=1)
                        val_acc += (pred == target).sum().item()
                
                # 保存验证结果
                val_acc_list.append(val_acc / test_num)
                val_epoch_list.append(e)
                print(f"epoch {e}/{num_epochs}  val_acc:{val_acc_list[-1]}")
                
                # 保存当前模型
                save_name = os.path.join(save_path, f"epoch_{e}_acc_{val_acc_list[-1]:.4f}.pth")
                save_dict = {
                    'state_dict': model.state_dict(), 
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list, 
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list
                }
                torch.save(save_dict, save_name)
                
    except Exception as exc:
        print(exc)
        
    finally:
        print(f'训练停止于epoch {e}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"训练时间: {train_time}")
    
    # 返回训练结果
    return {
        'loss_list': loss_list,
        'acc_list': acc_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'train_time': train_time
    }

def evaluate_model(model, data_loader, device):
    """
    评估模型性能
    
    参数:
        model: 训练好的模型
        data_loader: 数据加载器
        device: 计算设备
    
    返回:
        评估结果
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in tqdm(data_loader, desc="评估中"):
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    # 计算评估指标
    accuracy = accuracy_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds, average='binary')
    
    # 生成分类报告
    report = classification_report(all_targets, all_preds, target_names=['Negative', 'Positive'])
    
    return {
        'accuracy': accuracy,
        'recall': recall,
        'report': report,
        'predictions': all_preds,
        'targets': all_targets
    }

In [ ]:
def get_best_model(acc_list, epoch_list, save_path, del_others=True):
    """
    通过验证准确率找到最佳模型
    
    参数:
        acc_list: 准确率列表
        epoch_list: 对应的epoch列表
        save_path: 模型保存路径
        del_others: 是否删除其他模型
    
    返回:
        best_model_path: 最佳模型路径
    """
    acc_list = np.array(acc_list)
    epoch_list = np.array(epoch_list)
    best_index = np.argwhere(acc_list == np.max(acc_list))[-1].item()
    best_epoch = epoch_list[best_index]
    best_acc = acc_list[best_index]
    file_name = f"epoch_{best_epoch}_acc_{best_acc:.4f}.pth"
    best_model_path = os.path.join(save_path, file_name)
    print(f"最佳模型: {file_name}")
    
    # 删除其他模型
    if del_others:
        for f in os.listdir(save_path):
            if f.endswith('.pth') and os.path.join(save_path, f) != best_model_path:
                os.remove(os.path.join(save_path, f))
    
    return best_model_path

In [ ]:
# 选择要训练的标签ID
LABEL_ID = 1  # 可以根据需要修改

# 获取该标签的数据集
dataset_dict = get_dataset_for_label(LABEL_ID, all_data_dict, TRAIN_RATE)

# 创建训练、测试和验证数据集
train_dataset = BrainVoxelDataset(dataset_dict['train_samples'], dataset_dict['train_labels'])
test_dataset = BrainVoxelDataset(dataset_dict['test_samples'], dataset_dict['test_labels'])
val_dataset = BrainVoxelDataset(dataset_dict['val_samples'], dataset_dict['val_labels'])

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 设置计算设备
device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建模型
feature_dim = dataset_dict['train_samples'].shape[1]
model = BrainVoxelKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)

# 打印模型结构
summary(model)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# 训练模型
training_results = train_brain_voxel_kan(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=EPOCH,
    val_epoch=VAL_EPOCH,
    save_path=SAVE_PATH
)

# 获取最佳模型并评估
best_model_path = get_best_model(
    training_results['val_acc_list'],
    training_results['val_epoch_list'],
    SAVE_PATH
)

# 加载最佳模型
best_model = BrainVoxelKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)
best_model.load_state_dict(torch.load(best_model_path)['state_dict'])

# 在验证集上评估
print("在验证集上评估最佳模型...")
validation_results = evaluate_model(best_model, val_loader, device)
print(f"验证集准确率: {validation_results['accuracy']}")
print(f"验证集召回率: {validation_results['recall']}")
print("\n分类报告:")
print(validation_results['report'])

# 保存训练过程图表
plt.figure(figsize=(12, 5))

# 绘制损失曲线
plt.subplot(1, 2, 1)
plt.plot(range(len(training_results['loss_list'])), training_results['loss_list'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# 绘制准确率曲线
plt.subplot(1, 2, 2)
plt.plot(range(len(training_results['acc_list'])), training_results['acc_list'], label='Train Acc')
plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, f'training_curves_label_{LABEL_ID}.png'))
plt.show()

# 保存验证结果
validation_report = f"""
# 验证报告 - 标签 {LABEL_ID}

## 训练信息
- 训练时间: {training_results['train_time']:.2f} 秒
- 总训练轮数: {len(training_results['loss_list'])}
- 最佳模型: {os.path.basename(best_model_path)}
- 学习率: {LR}
- 批量大小: {BATCH_SIZE}
- 固定网格大小: {FIXED_GRID}

## 性能指标
- 验证集准确率: {validation_results['accuracy']:.4f}
- 验证集召回率: {validation_results['recall']:.4f}

## 分类报告
{validation_results['report']}
"""

with open(os.path.join(SAVE_PATH, f'validation_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(validation_report)

print(f"验证报告已保存至: {os.path.join(SAVE_PATH, f'validation_report_label_{LABEL_ID}.txt')}")

In [ ]:
def predict_whole_dataset(model, label_id, all_data_dict, device, batch_size=64):
    """
    使用训练好的模型对全部数据集进行预测
    
    参数:
        model: 训练好的模型
        label_id: 标签ID
        all_data_dict: 数据集字典
        device: 计算设备
        batch_size: 批处理大小
    
    返回:
        预测结果
    """
    # 获取完整标签集
    train_label_dir = all_data_dict['train_label_dir']
    val_label_dir = all_data_dict['val_label_dir']
    
    # 使用与get_dataset_for_label相同的方法获取数据
    def get_label_file_path(label_id, is_validation=False):
        if is_validation:
            pattern = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
        else:
            pattern = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
        
        matches = glob.glob(pattern)
        return matches[0] if matches else None
    
    # 收集所有数据
    all_samples = []
    all_labels = []
    
    # 获取目标标签的数据（正样本）
    train_file = get_label_file_path(label_id, False)
    val_file = get_label_file_path(label_id, True)
    
    if train_file:
        train_samples = np.load(train_file)
        train_labels = np.ones(len(train_samples))
        all_samples.append(train_samples)
        all_labels.append(train_labels)
    
    if val_file:
        val_samples = np.load(val_file)
        val_labels = np.ones(len(val_samples))
        all_samples.append(val_samples)
        all_labels.append(val_labels)
    
    # 获取其他标签的数据（负样本）
    valid_labels = all_data_dict['valid_labels']
    other_labels = [l for l in valid_labels if l != label_id]
    
    for other_label in other_labels:
        other_train_file = get_label_file_path(other_label, False)
        other_val_file = get_label_file_path(other_label, True)
        
        if other_train_file:
            other_train_samples = np.load(other_train_file)
            other_train_labels = np.zeros(len(other_train_samples))
            all_samples.append(other_train_samples)
            all_labels.append(other_train_labels)
        
        if other_val_file:
            other_val_samples = np.load(other_val_file)
            other_val_labels = np.zeros(len(other_val_samples))
            all_samples.append(other_val_samples)
            all_labels.append(other_val_labels)
    
    # 合并所有数据
    all_samples = np.vstack(all_samples)
    all_labels = np.concatenate(all_labels)
    
    # 创建数据集和加载器
    full_dataset = BrainVoxelDataset(all_samples, all_labels)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)
    
    # 进行预测
    model.eval()
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for data, _ in tqdm(full_loader, desc="全数据集预测"):
            data = data.to(device)
            output = model(data)
            probs = torch.softmax(output, dim=1)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # 保存正类的概率
    
    # 计算评估指标
    accuracy = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds, average='binary')
    
    # 计算AUC-PR（精确率-召回率曲线下面积）
    precision, recall_curve_points, _ = precision_recall_curve(all_labels, all_probs)
    auc_pr = auc(recall_curve_points, precision)
    
    # 生成分类报告
    report = classification_report(all_labels, all_preds, target_names=['Negative', 'Positive'])
    
    return {
        'accuracy': accuracy,
        'recall': recall,
        'auc_pr': auc_pr,
        'report': report,
        'predictions': all_preds,
        'probabilities': all_probs,
        'labels': all_labels,
        'samples': all_samples
    }

# 可视化精确率-召回率曲线
def plot_precision_recall_curve(labels, probabilities, save_path=None):
    """
    绘制精确率-召回率曲线
    
    参数:
        labels: 真实标签
        probabilities: 预测为正类的概率
        save_path: 保存路径，None表示不保存
    """
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    auc_pr = auc(recall, precision)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, lw=2, label=f'PR Curve (AUC = {auc_pr:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc='lower left')
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()

In [ ]:
# 在全数据集上应用最佳模型
print("在全数据集上进行预测...")
full_prediction_results = predict_whole_dataset(
    best_model, 
    LABEL_ID, 
    all_data_dict, 
    device, 
    BATCH_SIZE
)

# 打印性能指标
print(f"全数据集准确率: {full_prediction_results['accuracy']:.4f}")
print(f"全数据集召回率: {full_prediction_results['recall']:.4f}")
print(f"全数据集AUC-PR: {full_prediction_results['auc_pr']:.4f}")
print("\n全数据集分类报告:")
print(full_prediction_results['report'])

# 绘制精确率-召回率曲线
plot_precision_recall_curve(
    full_prediction_results['labels'],
    full_prediction_results['probabilities'],
    os.path.join(SAVE_PATH, f'pr_curve_label_{LABEL_ID}.png')
)

# 保存全数据集评估结果
full_dataset_report = f"""
# 全数据集评估报告 - 标签 {LABEL_ID}

## 性能指标
- 准确率: {full_prediction_results['accuracy']:.4f}
- 召回率: {full_prediction_results['recall']:.4f}
- AUC-PR: {full_prediction_results['auc_pr']:.4f}
- 正样本数量: {np.sum(full_prediction_results['labels'] == 1)}
- 负样本数量: {np.sum(full_prediction_results['labels'] == 0)}
- 预测为正的样本数: {np.sum(full_prediction_results['predictions'] == 1)}
- 预测为负的样本数: {np.sum(full_prediction_results['predictions'] == 0)}

## 分类报告
{full_prediction_results['report']}
"""

with open(os.path.join(SAVE_PATH, f'full_dataset_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(full_dataset_report)

print(f"全数据集评估报告已保存至: {os.path.join(SAVE_PATH, f'full_dataset_report_label_{LABEL_ID}.txt')}")

In [ ]:
def analyze_kan_model(model, data_samples, save_path=None):
    """
    简单分析KAN模型的特征重要性
    
    参数:
        model: 训练好的KAN模型
        data_samples: 数据样本
        save_path: 保存路径，None表示不保存
    """
    # 获取模型输入层的权重
    # 注意：这是针对简化KAN模型的近似分析
    input_weights = model.kan.layers[0].base_linear.weight.data.cpu().numpy()
    
    # 计算特征的平均绝对权重值（简单的重要性度量）
    feature_importance = np.mean(np.abs(input_weights), axis=0)
    
    # 找出前20个最重要的特征
    top_n = 20
    top_indices = np.argsort(feature_importance)[-top_n:][::-1]
    top_importance = feature_importance[top_indices]
    
    # 可视化特征重要性
    plt.figure(figsize=(12, 8))
    plt.barh(range(top_n), top_importance, align='center')
    plt.yticks(range(top_n), [f'Feature {i}' for i in top_indices])
    plt.xlabel('Mean Absolute Weight')
    plt.title('Top 20 Feature Importance')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()
    
    # 保存特征重要性数据
    importance_data = {
        'feature_index': np.arange(len(feature_importance)),
        'importance': feature_importance
    }
    
    return importance_data

# 分析模型
print("分析模型特征重要性...")
importance_data = analyze_kan_model(
    best_model,
    full_prediction_results['samples'],
    os.path.join(SAVE_PATH, f'feature_importance_label_{LABEL_ID}.png')
)

# 保存特征重要性数据
np.save(os.path.join(SAVE_PATH, f'feature_importance_label_{LABEL_ID}.npy'), importance_data)

In [ ]:
# 总结训练和评估结果
summary_report = f"""
# 模型训练及评估总结 - 标签 {LABEL_ID}

## 模型和训练配置
- 模型: BrainVoxel_1DKAN
- 输入特征维度: {feature_dim}
- 隐藏层维度: 64
- 输出类别数: {NUM_CLASS}
- 固定网格大小: {FIXED_GRID}
- 批量大小: {BATCH_SIZE}
- 学习率: {LR}
- 权重衰减: {WEIGHT_DECAY}
- 训练轮数: {len(training_results['loss_list'])}
- 训练时间: {training_results['train_time']:.2f} 秒

## 训练集分割
- 原训练集分割比例: {TRAIN_RATE}:{TEST_RATE}
- 训练集样本数: {len(train_dataset)}
- 测试集样本数: {len(test_dataset)}
- 验证集样本数: {len(val_dataset)}
- 正类样本比例: 约 50%

## 性能指标
- 测试集最佳准确率: {max(training_results['val_acc_list']):.4f}
- 验证集准确率: {validation_results['accuracy']:.4f}
- 验证集召回率: {validation_results['recall']:.4f}
- 全数据集准确率: {full_prediction_results['accuracy']:.4f}
- 全数据集AUC-PR: {full_prediction_results['auc_pr']:.4f}

## 重要发现
- KAN模型在脑体素分类任务上表现良好
- 简化的训练流程提高了训练效率
- 固定网格大小简化了模型，同时保持了分类性能
- 特征重要性分析可以帮助理解模型决策

## 建议
- 可以尝试不同的网格大小以平衡性能和复杂度
- 考虑对最重要的特征进行进一步分析
- 将训练流程应用于其他标签
"""

with open(os.path.join(SAVE_PATH, f'summary_report_label_{LABEL_ID}.txt'), 'w') as f:
    f.write(summary_report)

print(f"训练和评估总结已保存至: {os.path.join(SAVE_PATH, f'summary_report_label_{LABEL_ID}.txt')}")
print("完成！")